In [1]:
import pandas as pd

df = pd.read_csv("bank-full.csv", sep=";")

print(df.head())
print(df.shape)

   age           job  marital  education default  balance housing loan  \
0   58    management  married   tertiary      no     2143     yes   no   
1   44    technician   single  secondary      no       29     yes   no   
2   33  entrepreneur  married  secondary      no        2     yes  yes   
3   47   blue-collar  married    unknown      no     1506     yes   no   
4   33       unknown   single    unknown      no        1      no   no   

   contact  day month  duration  campaign  pdays  previous poutcome   y  
0  unknown    5   may       261         1     -1         0  unknown  no  
1  unknown    5   may       151         1     -1         0  unknown  no  
2  unknown    5   may        76         1     -1         0  unknown  no  
3  unknown    5   may        92         1     -1         0  unknown  no  
4  unknown    5   may       198         1     -1         0  unknown  no  
(45211, 17)


In [29]:
import pandas as pd
import numpy as np

# load
# df = pd.read_csv("dat.csv")

# duplicates
df = df.drop_duplicates()

# constant columns
df = df.loc[:, df.nunique() > 1]

# missing handling
df = df.loc[:, df.isnull().mean() < 0.5]
df = df[df.isnull().mean(axis=1) < 0.3]

for col in df.columns:
    if df[col].isnull().sum() > 0:
        if df[col].dtype in ['int64', 'float64']:
            df[col].fillna(df[col].mean(), inplace=True)
        else:
            df[col].fillna(df[col].mode()[0], inplace=True)

# outliers (IQR)
num_cols = df.select_dtypes(include=np.number).columns

for col in num_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1

    df = df[(df[col] >= Q1 - 1.5 * IQR) & (df[col] <= Q3 + 1.5 * IQR)]

# save
df.to_csv("cleaned_data.csv", index=False)
print(df.shape)

(19327, 11)


In [16]:
from sklearn.preprocessing import LabelEncoder

for col in df.select_dtypes(include='object'):
    df[col] = LabelEncoder().fit_transform(df[col])

X = df.drop("y", axis=1)
y = df["y"]

In [17]:
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import LabelEncoder

X_chi = df.drop("y", axis=1)
X_chi = X_chi.apply(LabelEncoder().fit_transform)

chi_scaler = MinMaxScaler()
X_chi_scaled = chi_scaler.fit_transform(X_chi)

In [18]:
import numpy as np

corr = pd.DataFrame(X).corrwith(pd.Series(y))
pearson_features = corr.abs().sort_values(ascending=False).head(8).index
print("Pearson:", list(pearson_features))

Pearson: ['duration', 'contact', 'housing', 'pdays', 'previous', 'poutcome', 'campaign', 'loan']


In [19]:
from sklearn.feature_selection import mutual_info_classif

mi = mutual_info_classif(X, y, random_state=42)
mi_features = pd.Series(mi, index=X.columns).sort_values(ascending=False).head(8).index
print("Mutual Information:", list(mi_features))
info_gain_features = mi_features

Mutual Information: ['duration', 'poutcome', 'month', 'pdays', 'balance', 'housing', 'contact', 'previous']


In [20]:
spearman = pd.DataFrame(X).corrwith(pd.Series(y), method='spearman')
spearman_features = spearman.abs().sort_values(ascending=False).head(8).index
print("Spearman:", list(spearman_features))

Spearman: ['duration', 'previous', 'pdays', 'contact', 'poutcome', 'housing', 'balance', 'campaign']


In [21]:
from sklearn.feature_selection import chi2

chi_scores, _ = chi2(X_chi_scaled, y)
chi_features = pd.Series(chi_scores, index=X.columns).sort_values(ascending=False).head(8).index
print("Chi-square:", list(chi_features))

Chi-square: ['duration', 'contact', 'housing', 'pdays', 'loan', 'previous', 'balance', 'poutcome']


In [22]:
def entropy(col):
    probs = col.value_counts(normalize=True)
    return -np.sum(probs * np.log2(probs + 1e-9))

gain_ratio_scores = {}

for col in X.columns:
    ig = mutual_info_classif(X[[col]], y, random_state=42)[0]
    split_info = entropy(X[col])
    gain_ratio_scores[col] = ig / (split_info + 1e-9)

gain_ratio_features = pd.Series(gain_ratio_scores).sort_values(ascending=False).head(8).index
print("Gain Ratio:", list(gain_ratio_features))

Gain Ratio: ['poutcome', 'housing', 'contact', 'default', 'pdays', 'previous', 'month', 'loan']


In [24]:
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif

scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

selector = SelectKBest(score_func=f_classif, k=10)
X_selected = selector.fit_transform(X_scaled, y)

anova_features = X.columns[selector.get_support()]
print("ANOVA Features:", list(anova_features))

ANOVA Features: ['education', 'balance', 'housing', 'loan', 'contact', 'duration', 'campaign', 'pdays', 'previous', 'poutcome']


In [25]:
from collections import Counter

it = X.columns.tolist()
all_features = (
    list(anova_features) +
    list(pearson_features) +
    list(spearman_features) +
    list(mi_features) +
    list(chi_features) +
    list(gain_ratio_features) +
    list(it)
)

vote = Counter(all_features)

final_features = [f for f, count in vote.items() if count >= 2]
print("Final selected features:", final_features)


Final selected features: ['education', 'balance', 'housing', 'loan', 'contact', 'duration', 'campaign', 'pdays', 'previous', 'poutcome', 'month', 'default']


In [26]:
X = df[final_features]
y = df["y"]

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [27]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)